# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaheerkhan1117/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

**Finding 1: "Average Position (43%), Impressions (32%), and Scroll Depth (15%) are the top Random Forest predictors of Health Score" (ML Appendix — Feature Importance)**
- Where the label comes from: Health Score is a FlyRank composite explicitly defined as `Impressions (30pts) + Position (30pts) + CTR (20pts) + Scroll Depth (20pts)`. Three of the model's top four "predictors" are literal components of the formula being predicted — the paper itself notes this and calls the importances "descriptive rather than causal."
- Does the validation design carry the claim?: The methodology section only reports an 80/20 split for this Random Forest; there's no mention of a client/brand-grouped holdout across the 57 brands, and the paper is careful to only claim "model behavior," not causal or predictive value.
- My question, constructive tone: Given that position, impressions, and scroll depth are direct formula inputs to the label, was a version of this model trained *without* those three components tested — the same train-with/train-without check `w05_model` runs here in Section 2 — to see whether the 43/32/15 importance split collapses once the definitional overlap is removed, or whether some of it survives?

**Finding 2: "Logistic regression separates growing from declining content at 71% holdout accuracy" (ML Appendix — Growth & Classification)**
- Where the label comes from: `Trend Direction` is computed from the 30-day-vs-previous-30-day impression change (down = >10% decline) — the same shape of "compare two windows of the same metric" that produces `is_declining` in this lane's own data, just at a different threshold and window length.
- Does the validation design carry the claim?: Methodology states an 80/20 split for this model too, with no mention of grouping by brand. Finding #1 elsewhere in the paper reports roughly 74.8K growing vs. 45.6K declining pages in the same comparison — a ~62% majority class — but the 71% accuracy isn't reported next to that base rate anywhere.
- My question, constructive tone: Since pages from the same brand likely share baseline growth momentum (a brand doing well tends to have many growing pages at once), was the 80/20 split held out by brand rather than by content row? And separately, could the write-up report the majority-class baseline alongside the 71% so a reader can see the actual lift over just guessing "growing" every time?

In [5]:
# Base-rate context for Finding 2, computed from this paper's own reported numbers
# (up/down counts are quoted directly from Finding #1's table, not re-derived from private data)
up, down = 74_800, 45_600
majority_class_rate = max(up, down) / (up + down)
print(f"Growing vs declining base rate from the paper's own Finding #1 table: {majority_class_rate:.1%} majority class")
print(f"Reported logistic regression holdout accuracy: 71.0%")
print(f"Lift over majority-class baseline: {0.710 - majority_class_rate:+.1%}")
print("\nThis is the same 'always print the base rate' check from hunting-leakage-and-validating —")
print("a 71% accuracy against a ~62% base rate is real but modest lift, not the headline number alone.")

Growing vs declining base rate from the paper's own Finding #1 table: 62.1% majority class
Reported logistic regression holdout accuracy: 71.0%
Lift over majority-class baseline: +8.9%

This is the same 'always print the base rate' check from hunting-leakage-and-validating —
a 71% accuracy against a ~62% base rate is real but modest lift, not the headline number alone.


## 2. My model under an honest split (before/after)

`w05_model` already used a client-grouped split as its primary design, reasoned out in that
notebook's Section 2. The honest "before" here is the split most people reach for first — a
plain random row-level split, the same shape `w03_data_contract`'s quick demo used — rebuilt
side by side with the grouped "after" so the gap between them is a real, computed number instead
of an assertion.

In [6]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/shaheerkhan1117/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":    f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":    f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
MONTH = "2026-03"

raw = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions)                                            AS total_impressions,
        SUM(gsc_clicks)                                                 AS total_clicks,
        AVG(gsc_avg_position)                                           AS avg_position,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)  AS active_days,
        STDDEV_SAMP(gsc_avg_position)                                   AS position_volatility,
        MODE(ga4_data_available)                                       AS ga4_data_available,
        SUM(CASE WHEN report_date < DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date >= DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
    GROUP BY 1, 2
    HAVING SUM(CASE WHEN report_date < DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) > 0
""").df()

df = raw.copy()
df["ctr"] = (df["total_clicks"] / df["total_impressions"]).round(4)
df["impressions_per_active_day"] = (df["total_impressions"] / df["active_days"]).round(2)
df["volatility_is_filled"] = df["position_volatility"].isna().astype(int)
df["position_volatility"] = df["position_volatility"].fillna(0.0)
ga4_dummies = pd.get_dummies(df["ga4_data_available"], prefix="ga4", dummy_na=True)
df = pd.concat([df, ga4_dummies], axis=1)
df["pct_change"] = (df["imp_second_half"] - df["imp_first_half"]) / df["imp_first_half"]
df["is_declining"] = (df["pct_change"] < -0.2).astype(int)

FINAL_FEATURES = (["total_impressions", "total_clicks", "avg_position", "active_days",
                    "position_volatility", "volatility_is_filled", "ctr",
                    "impressions_per_active_day"] + list(ga4_dummies.columns))

def fit_and_score(train_df, test_df):
    scaler = StandardScaler().fit(train_df[FINAL_FEATURES])
    clf = LogisticRegression(max_iter=1000).fit(
        scaler.transform(train_df[FINAL_FEATURES]), train_df["is_declining"])
    return roc_auc_score(test_df["is_declining"],
                          clf.predict_proba(scaler.transform(test_df[FINAL_FEATURES]))[:, 1])

# BEFORE — plain random row-level split, same shape as w03_data_contract's quick demo
before_train, before_test = train_test_split(df, test_size=0.3, random_state=42, stratify=df["is_declining"])
before_auc = fit_and_score(before_train, before_test)

# AFTER — client-grouped split, same design as w05_model
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_hash_id"]))
after_train, after_test = df.iloc[train_idx], df.iloc[test_idx]
after_auc = fit_and_score(after_train, after_test)

print(f"BEFORE — random split AUC:          {before_auc:.3f}")
print(f"AFTER  — client-grouped split AUC:  {after_auc:.3f}")
print(f"Gap:                                 {before_auc - after_auc:+.3f}")
print("\nA gap here (before > after) means the random split was letting the model partly learn")
print("client identity rather than a signal that generalizes to a client it hasn't seen.")

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

BEFORE — random split AUC:          0.672
AFTER  — client-grouped split AUC:  0.603
Gap:                                 +0.069

A gap here (before > after) means the random split was letting the model partly learn
client identity rather than a signal that generalizes to a client it hasn't seen.


## 3. Leakage audit

Same hunt as `w03_feature_leakage_check`, re-run on the final feature set actually used to
train — `FINAL_FEATURES` above, evaluated on the grouped test split from Section 2 so the
numbers reflect the same honest split the model itself was scored on.

In [7]:
checked = after_test.copy()

# Attack 1 — single-feature AUC scan, nothing should sit near 1.0
single_feature_auc = {}
for col in FINAL_FEATURES:
    d = checked.dropna(subset=[col, "is_declining"])
    if d[col].nunique() < 2:
        continue
    auc = roc_auc_score(d["is_declining"], d[col])
    single_feature_auc[col] = round(max(auc, 1 - auc), 3)

print("Single-feature AUC against is_declining, grouped test split:")
print(pd.Series(single_feature_auc).sort_values(ascending=False))

# Attack 2 — the same trap, re-run on the final feature set and the honest grouped split
checked["second_half_share"] = checked["imp_second_half"] / (checked["imp_first_half"] + checked["imp_second_half"])

def quick_auc(cols, train_data, test_data):
    scaler = StandardScaler().fit(train_data[cols])
    clf = LogisticRegression(max_iter=1000).fit(scaler.transform(train_data[cols]), train_data["is_declining"])
    return roc_auc_score(test_data["is_declining"], clf.predict_proba(scaler.transform(test_data[cols]))[:, 1])

after_train_trap = after_train.copy()
after_train_trap["second_half_share"] = (
    after_train_trap["imp_second_half"] / (after_train_trap["imp_first_half"] + after_train_trap["imp_second_half"]))

print(f"\nHonest AUC (final feature set, grouped split): {after_auc:.3f}")
leaked_auc = quick_auc(FINAL_FEATURES + ["second_half_share"], after_train_trap, checked)
print(f"Leaked AUC (adding second_half_share back in):  {leaked_auc:.3f}")
print("\nsecond_half_share deleted — it only ever existed to confirm the leak is still catchable.")
del checked["second_half_share"]

Single-feature AUC against is_declining, grouped test split:
total_impressions             0.596
impressions_per_active_day    0.585
total_clicks                  0.577
active_days                   0.563
ctr                           0.557
ga4_True                      0.537
ga4_False                     0.536
volatility_is_filled          0.521
position_volatility           0.502
avg_position                  0.501
ga4_<NA>                      0.501
dtype: float64

Honest AUC (final feature set, grouped split): 0.603
Leaked AUC (adding second_half_share back in):  1.000

second_half_share deleted — it only ever existed to confirm the leak is still catchable.


## 4. Claim rewrite

**Boldest draft sentence, the kind that's easy to write after a table like Section 2's:**
> "This model predicts which pages will decline, and the grouped split proves it generalizes to
> new clients."

**Problems with it:** "predicts... will decline" claims a forecast this single-month,
within-month label was never validated to make — there's no next-month data behind that verb.
"Proves it generalizes" is doing more work than one grouped split on one month can support; it's
one piece of evidence, not proof, and it says nothing about generalizing across time, only
across clients.

**Rewrite, in the paper's required register:**
> "On this month's data, the model's score is directionally associated with within-month
> decline as observed in a client-grouped holdout — a stronger check than a random split, but
> still a single month and a single split. Treat the score as decision-support for prioritizing
> review, not as a forecast of future performance."`

In [8]:
claims = pd.DataFrame([
    {"version": "draft (bold)",
     "text": "This model predicts which pages will decline, and the grouped split proves it generalizes to new clients."},
    {"version": "rewrite (safe)",
     "text": ("On this month's data, the model's score is directionally associated with within-month decline as "
              "observed in a client-grouped holdout — a stronger check than a random split, but still a single "
              "month and a single split. Treat the score as decision-support for prioritizing review, not as a "
              "forecast of future performance.")},
])
pd.set_option("display.max_colwidth", None)
claims

,version,text
0,draft (bold),"This model predicts which pages will decline, and the grouped split proves it generalizes to new clients."
1,rewrite (safe),"On this month's data, the model's score is directionally associated with within-month decline as observed in a client-grouped holdout — a stronger check than a random split, but still a single month and a single split. Treat the score as decision-support for prioritizing review, not as a forecast of future performance."


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.